In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

**Load the Dataset**

In [ ]:
data = pd.read_csv("CSE_Skill_Gap_Dataset_1000.csv")

print("Dataset shape:", data.shape)
data.head()

In [ ]:
data.info()

In [ ]:
print(data.isnull().sum())

**Select Relevant Features**

In [ ]:
# Select Relevant Features

features = [
    "StudentName",
    "CurriculumCoverage",
    "IndustryDemand",
    "JobFrequency",
    "ExpertImportance",
    "Programming",
    "DataStructures",
    "Algorithms",
    "Database",
    "WebDevelopment",
    "Cloud",
    "AI_ML",
    "Communication",
    "ProblemSolving",
    "Teamwork",
    "Projects",
    "Internship",
    "Certifications",
    "CurriculumAlignment",
    "IndustryRequirement",
    "SkillGap"
]

# Target Variable

target = "GapStatus"

# Create Feature Matrix and Target Vector

X = data[features]
y = data[target]

# Display Features and Target

print(X.head())
print(y.head())

**Split the Dataset**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


**Before Proper Preprocessing**

In [ ]:
# Before Proper Preprocessing

numerical_only = [
    "CurriculumCoverage",
    "IndustryDemand",
    "JobFrequency",
    "ExpertImportance",
    "Programming",
    "DataStructures",
    "Algorithms",
    "Database",
    "WebDevelopment",
    "Cloud",
    "AI_ML",
    "Communication",
    "ProblemSolving",
    "Teamwork",
    "Projects",
    "Internship",
    "Certifications",
    "CurriculumAlignment",
    "IndustryRequirement",
    "SkillGap"
]

baseline_train = X_train[numerical_only].copy()
baseline_test = X_test[numerical_only].copy()

train_valid_rows = baseline_train.dropna().index
test_valid_rows = baseline_test.dropna().index

baseline_train = baseline_train.loc[train_valid_rows]
baseline_test = baseline_test.loc[test_valid_rows]

y_train_baseline = y_train.loc[train_valid_rows]
y_test_baseline = y_test.loc[test_valid_rows]

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(baseline_train, y_train_baseline)

baseline_predictions = baseline_model.predict(baseline_test)

baseline_accuracy = accuracy_score(
    y_test_baseline,
    baseline_predictions
)

print("Baseline accuracy:", baseline_accuracy)
print("Training records used:", len(baseline_train))
print("Testing records used:", len(baseline_test))

**Define Numerical and Categorical Features**

In [ ]:
# Define Numerical and Categorical Features

numerical_features = [
    "CurriculumCoverage",
    "IndustryDemand",
    "JobFrequency",
    "ExpertImportance",
    "Programming",
    "DataStructures",
    "Algorithms",
    "Database",
    "WebDevelopment",
    "Cloud",
    "AI_ML",
    "Communication",
    "ProblemSolving",
    "Teamwork",
    "Projects",
    "Internship",
    "Certifications",
    "CurriculumAlignment",
    "IndustryRequirement",
    "SkillGap"
]

categorical_features = [
    "StudentName"
]

**Numerical Preprocessing**
For numerical variables:

Replace missing values using the median.
Standardize the values.

In [ ]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

**Categorical Preprocessing**

For categorical variables:

Replace missing values using the most frequent category.
Convert categories using one-hot encoding.

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

**Combine the Preprocessing Steps**

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

**Create the Complete Machine-Learning Pipeline**

In [ ]:
complete_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

In [ ]:
complete_pipeline.fit(X_train, y_train)

In [ ]:
processed_predictions = complete_pipeline.predict(X_test)

processed_accuracy = accuracy_score(
    y_test,
    processed_predictions
)

print("Accuracy after preprocessing:", processed_accuracy)

In [ ]:
print(
    classification_report(
        y_test,
        processed_predictions
    )
)

**Compare Before and After Preprocessing**

In [ ]:
comparison = pd.DataFrame(
    {
        "Approach": [
            "Before proper preprocessing",
            "After preprocessing pipeline"
        ],
        "Accuracy": [
            baseline_accuracy,
            processed_accuracy
        ],
        "Training records used": [
            len(baseline_train),
            len(X_train)
        ],
        "Testing records used": [
            len(baseline_test),
            len(X_test)
        ]
    }
)

print(comparison)

**Generate the Processed Dataset**

In [ ]:
X_train_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_train)

X_test_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_test)

In [ ]:
feature_names = complete_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

print(feature_names)

In [ ]:
processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

processed_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [ ]:
processed_train_df["GapStatus"] = y_train.values
processed_test_df["GapStatus"] = y_test.values

In [ ]:
processed_train_df.head()

**Save the Processed Dataset**

In [ ]:
processed_train_df.to_csv(
    "Skill_Gap_processed_train.csv",
    index=False
)

processed_test_df.to_csv(
    "Skill_Gap_processed_test.csv",
    index=False
)

In [ ]:
from google.colab import files

files.download("Skill_Gap_processed_train.csv")
files.download("Skill_Gap_processed_test.csv")

**Save the Complete Pipeline**

In [ ]:
joblib.dump(
    complete_pipeline,
    "Skill_Gap_preprocessing_model_pipeline.pkl"
)

In [ ]:
files.download(
    "Skill_Gap_preprocessing_model_pipeline.pkl"
)

**Reuse the Saved Pipeline**

In [ ]:
loaded_pipeline = joblib.load(
    "Skill_Gap_preprocessing_model_pipeline.pkl"
)

In [ ]:
new_student = pd.DataFrame(
    {
        "StudentName": ["Meghana Reddy"],
        "CurriculumCoverage": [75],
        "IndustryDemand": [90],
        "JobFrequency": [8],
        "ExpertImportance": [5],
        "Programming": [85],
        "DataStructures": [80],
        "Algorithms": [78],
        "Database": [82],
        "WebDevelopment": [76],
        "Cloud": [74],
        "AI_ML": [88],
        "Communication": [84],
        "ProblemSolving": [90],
        "Teamwork": [86],
        "Projects": [4],
        "Internship": [1],
        "Certifications": [3],
        "CurriculumAlignment": [76],
        "IndustryRequirement": [90],
        "SkillGap": [14]
    }
)

In [ ]:
prediction = loaded_pipeline.predict(new_student)

if prediction[0] == 0:
    print("Prediction: Aligned")
else:
    print("Prediction: Gap Identified")